# Phase 9 — IPDK 및 레시피 배포 (예제)

검증된 측정 코드를 '배포 가능한 레시피'로 다듬는 과정을 익힙니다. 핵심: 함수 하나로 묶기 · 설정 분리 · dict 반환 · 예외 처리 · 재현성.

## 1. 예제 이미지 (알려진 정답: 35px = 8.75nm)

In [ ]:
import numpy as np
rng=np.random.default_rng(3)
img=np.full((220,360),90,float); img[95:130,:]=185
img=np.clip(img+rng.normal(0,8,img.shape),0,255).astype(np.uint8)
print('참 두께 35px =', 35*0.25, 'nm')

## 2. 배포 가능한 measure() 만들기
노트북에 흩어진 측정 로직을 표준 시그니처 `measure(image, scale, params)` 로 묶습니다. 설정은 `DEFAULT_PARAMS` 로 분리하고, 결과는 dict 로 반환합니다.

In [ ]:
DEFAULT_PARAMS = {'n_positions': 20, 'smooth': 7, 'min_edges': 2}

def _profile(image, x, smooth):
    col = image[:, x].astype(float)
    if smooth and smooth >= 3:
        pad = smooth//2
        cp = np.pad(col, pad, mode='edge')          # 0 패딩 금지 (레벨 왜곡 방지)
        col = np.convolve(cp, np.ones(smooth)/smooth, mode='valid')
    return col

def _edges(prof):
    level = (np.percentile(prof,5)+np.percentile(prof,95))/2
    xs=[]
    for i in range(len(prof)-1):
        y0,y1=prof[i],prof[i+1]
        if (y0-level)*(y1-level)<0:
            xs.append(i+(level-y0)/(y1-y0))
    return xs

def measure(image, scale, params=None):
    p={**DEFAULT_PARAMS, **(params or {})}
    try:
        if image is None or image.ndim!=2:
            return {'status':'fail','error':'2D 이미지 아님'}
        H,W=image.shape
        xs=np.linspace(int(W*0.1),int(W*0.9),p['n_positions']).astype(int)
        res=[]
        for x in xs:
            e=_edges(_profile(image,x,p['smooth']))
            if len(e)<p['min_edges']:
                continue                 # 실패는 건너뜀
            res.append((e[-1]-e[0])*scale)
        if not res:
            return {'status':'fail','error':'유효 측정 없음'}
        res=np.array(res)
        return {'mean_nm':float(res.mean()),'std_nm':float(res.std()),
                'n':int(res.size),'unit':'nm','status':'ok'}
    except Exception as e:
        return {'status':'fail','error':str(e)}

measure(img, 0.25)

## 3. 배포 전 테스트 — 정답과 대조 + 재현성
알려진 시료로 정답(8.75nm)과 대조하고, 같은 입력에 같은 출력이 나오는지(재현성) 확인합니다.

In [ ]:
r = measure(img, 0.25)
print(r)

assert r['status']=='ok'
assert abs(r['mean_nm']-8.75) < 0.3          # 정답과 대조
assert measure(img,0.25)==r                  # 같은 입력 -> 같은 출력
print('검증 통과')

## 4. 실패 입력도 안전하게
배포 코드는 실패해도 멈추지 않고 status 를 반환해야 합니다.

In [ ]:
print(measure(None, 0.25))                   # 잘못된 입력
print(measure(np.zeros((50,50),np.uint8), 0.25))  # 층이 없어 측정 불가

## 5. 설정을 바꿔 재사용 (파라미터 분리의 이점)
코드를 고치지 않고 파라미터만 바꿔 다른 조건으로 측정합니다.

In [ ]:
print('위치 5개 :', measure(img, 0.25, {'n_positions':5})['n'])
print('위치 40개:', measure(img, 0.25, {'n_positions':40})['n'])

## 6. 레시피 파일로 분리
완성한 measure() 는 `09_recipe_template.py` 처럼 별도 .py 파일로 두고, 버전·측정 방법을 기록합니다. IPDK 의 RCP 패키징·등록 절차는 사내 IPDK 문서를 따릅니다.

```python
# 사용 예 (IPDK 연동 골격)
# from recipe import measure
# result = measure(image, scale, params)   # IPDK 가 입력을 주입
# context.report(result)                   # IPDK 로 결과 반환
```

---
예제 실행을 마친 후 `09_practice.ipynb` 의 문제를 해결하십시오.